# Chapter 11: Volatility Spreads in code

Rechecks the numbers in Chapter 11 (butterfly and condor values, the 3 x 1 ratio spread, the calendar-spread tables, interest-rate and dividend effects, the Greek totals of every example spread, and the choice of spread when implied volatility is low or high) and draws the figures used in the notes.
Notes: [[chapters/11-volatility-spreads]]

Only numpy and matplotlib are needed (the normal curve uses `math.erf`). Prices and amounts follow the book. Anything marked *our* is not in the book.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def N(x):
    # cumulative standard normal
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def bs_price(S, K, T, r, vol, kind="c", q=0.0):
    # Black-Scholes value of a European call (kind="c") or put (kind="p"); T in years
    if T <= 0:
        return max(S - K, 0.0) if kind == "c" else max(K - S, 0.0)
    d1 = (math.log(S / K) + (r - q + 0.5 * vol * vol) * T) / (vol * math.sqrt(T))
    d2 = d1 - vol * math.sqrt(T)
    if kind == "c":
        return S * math.exp(-q * T) * N(d1) - K * math.exp(-r * T) * N(d2)
    return K * math.exp(-r * T) * N(-d2) - S * math.exp(-q * T) * N(-d1)

def bs_delta(S, K, T, r, vol, kind="c", q=0.0):
    d1 = (math.log(S / K) + (r - q + 0.5 * vol * vol) * T) / (vol * math.sqrt(T))
    return math.exp(-q * T) * (N(d1) if kind == "c" else N(d1) - 1)

def bs_gamma_vega(S, K, T, r, vol, q=0.0):
    d1 = (math.log(S / K) + (r - q + 0.5 * vol * vol) * T) / (vol * math.sqrt(T))
    pdf = math.exp(-0.5 * d1 * d1) / math.sqrt(2 * math.pi)
    gamma = math.exp(-q * T) * pdf / (S * vol * math.sqrt(T))
    vega = S * math.exp(-q * T) * pdf * math.sqrt(T)   # per 1.00 of volatility
    return gamma, vega

def curve(f, xs):
    return np.array([f(x) for x in xs])

def call_val(S, K):
    return np.maximum(np.asarray(S, dtype=float) - K, 0.0)

def put_val(S, K):
    return np.maximum(K - np.asarray(S, dtype=float), 0.0)

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25})

## 1. Butterfly and condor at expiration (pages 174-177)
A long call butterfly 90/100/110 is $+1$ of the 90 call, $-2$ of the 100 call, $+1$ of the 110 call. The book's table: underlying 80, 100, 120 gives 0, +10, 0. A long condor (90/95/105/110, one of each) is worth at most the gap between the two lower strikes (5) and at least 0.

In [ ]:
S = np.array([80, 100, 120])
fly = call_val(S, 90) - 2 * call_val(S, 100) + call_val(S, 110)
print("butterfly values at 80, 100, 120:", fly, " (book 0, +10, 0)")

S2 = np.array([85, 92, 97, 100, 103, 108, 115])
condor = call_val(S2, 90) - call_val(S2, 95) - call_val(S2, 105) + call_val(S2, 110)
for s, v in zip(S2, condor):
    print(f"underlying {s:>4}: condor worth {v:.1f}")

Sx = np.linspace(75, 125, 501)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
legs = [(+1, 90), (-2, 100), (+1, 110)]
for qty, K in legs:
    axs[0].plot(Sx, qty * call_val(Sx, K), lw=1, ls="--", label=f"{qty:+d} x {K} call")
axs[0].plot(Sx, call_val(Sx, 90) - 2 * call_val(Sx, 100) + call_val(Sx, 110), "k", lw=3, label="long butterfly")
axs[0].set_title("Long butterfly 90/100/110 at expiration"); axs[0].legend(fontsize=7)
legs = [(+1, 90), (-1, 95), (-1, 105), (+1, 110)]
for qty, K in legs:
    axs[1].plot(Sx, qty * call_val(Sx, K), lw=1, ls="--", label=f"{qty:+d} x {K} call")
axs[1].plot(Sx, call_val(Sx, 90) - call_val(Sx, 95) - call_val(Sx, 105) + call_val(Sx, 110), "k", lw=3, label="long condor")
axs[1].set_title("Long condor 90/95/105/110 at expiration"); axs[1].legend(fontsize=7)
for a in axs:
    a.axhline(0, color="grey", lw=0.8); a.set_xlabel("underlying price at expiration"); a.set_ylabel("value of the position")
plt.tight_layout(); plt.savefig("../images/ch11-butterfly-condor.png", dpi=130); plt.show()

## 2. A delta-neutral ratio straddle (page 171)
Underlying 100. The September 95 call is in the money (delta 75), the September 95 put is out of the money (delta $-25$). One call and one put gives $75 - 25 = +50$ deltas (a bull straddle). To be delta neutral we need $75 = 3 \times 25$, so three puts per call.

In [ ]:
call_delta, put_delta = 75, -25
print("1 call + 1 put :", call_delta + put_delta, "deltas")
print("1 call + 3 puts:", call_delta + 3 * put_delta, "deltas")

## 3. The 3 x 1 call ratio spread (page 179)
Underlying 100. Buy 3 October 105 calls at 1.00 (delta 25 each), sell 1 October 95 call at 6.00 (delta 75). Credit $= 6.00 - 3 \times 1.00 = 3.00$. We check the three P&L rows in the book, then find the breakeven prices and draw the legs.

Between 95 and 105 only the short call is "alive", so the position loses 1 for every point the underlying rises. Above 105 the three long calls are alive too: slope $= -1 + 3 = +2$.

In [ ]:
def ratio_pnl(S, buy_qty, buy_k, buy_px, sell_qty, sell_k, sell_px):
    # expiration P&L of a call ratio spread, including the premium paid or received
    S = np.asarray(S, dtype=float)
    return (buy_qty * (call_val(S, buy_k) - buy_px)
            - sell_qty * (call_val(S, sell_k) - sell_px))

print("delta:", 3 * 25 - 1 * 75, " credit:", 6.00 - 3 * 1.00)
for s in (80, 120, 100):
    print(f"underlying {s}: total P&L {float(ratio_pnl(s, 3, 105, 1.00, 1, 95, 6.00)):+.2f}")
print("book: +3.00, +23.00, -2.00")

# breakevens: between 95 and 105 the P&L is 3 - (S - 95); above 105 it is 2S - 217
be_low = 95 + 3.00
be_high = 217 / 2
print("breakevens:", be_low, be_high, " worst loss at 105:", float(ratio_pnl(105, 3, 105, 1.00, 1, 95, 6.00)))

In [ ]:
Sx = np.linspace(80, 125, 451)
fig, axs = plt.subplots(1, 2, figsize=(10.5, 4))

# left: call ratio backspread (buy 3 / sell 1)
axs[0].plot(Sx, 3 * (call_val(Sx, 105) - 1.00), "--", lw=1, label="3 long 105 calls (paid 1.00 each)")
axs[0].plot(Sx, -(call_val(Sx, 95) - 6.00), "--", lw=1, label="1 short 95 call (received 6.00)")
tot = ratio_pnl(Sx, 3, 105, 1.00, 1, 95, 6.00)
axs[0].plot(Sx, tot, "k", lw=3, label="backspread: buy 3 / sell 1")
axs[0].set_title("Call ratio spread, buy more than sell (backspread)")

# right: frontspread (sell 3 / buy 1) is the mirror image
axs[1].plot(Sx, -3 * (call_val(Sx, 105) - 1.00), "--", lw=1, label="3 short 105 calls")
axs[1].plot(Sx, (call_val(Sx, 95) - 6.00), "--", lw=1, label="1 long 95 call")
axs[1].plot(Sx, -tot, "k", lw=3, label="frontspread: sell 3 / buy 1")
axs[1].set_title("Call ratio spread, sell more than buy (frontspread)")
for a in axs:
    a.axhline(0, color="grey", lw=0.8); a.set_ylim(-35, 35)
    a.set_xlabel("underlying price at expiration"); a.set_ylabel("profit or loss"); a.legend(fontsize=7, loc="upper left")
axs[0].annotate("credit 3.00 kept\nif underlying <= 95", (82, 3), (83, 14), fontsize=8, arrowprops=dict(arrowstyle="->"))
axs[0].annotate("worst loss -7 at 105", (105, -7), (106, -22), fontsize=8, arrowprops=dict(arrowstyle="->"))
plt.tight_layout(); plt.savefig("../images/ch11-ratio-spread-legs.png", dpi=130); plt.show()

## 4. Why the delta-neutral ratio is the ratio of the deltas, and is it always a credit? (page 179)
For a call ratio spread the number of options at the second strike is $n = \Delta_1 / \Delta_2$. The book says that under a traditional model a delta-neutral spread that buys more than it sells should always be a credit. We test that for every pair of strikes between 80 and 130 using the Figure 11-31 inputs (stock 100, two months, 20%, 6% interest, no dividend). *Our check, not in the book.*

In [ ]:
S0, T, r, vol = 100, 2 / 12, 0.06, 0.20
strikes = range(80, 131)
worst_call, worst_put = None, None
for k1 in strikes:
    for k2 in strikes:
        if k2 <= k1:
            continue
        # call: sell one at the lower strike k1 (bigger delta), buy n at the higher strike k2
        n = bs_delta(S0, k1, T, r, vol, "c") / bs_delta(S0, k2, T, r, vol, "c")
        credit = bs_price(S0, k1, T, r, vol, "c") - n * bs_price(S0, k2, T, r, vol, "c")
        if worst_call is None or credit < worst_call[0]:
            worst_call = (credit, k1, k2, n)
        # put: sell one at the higher strike k2 (bigger |delta|), buy n at the lower strike k1
        n = bs_delta(S0, k2, T, r, vol, "p") / bs_delta(S0, k1, T, r, vol, "p")
        credit = bs_price(S0, k2, T, r, vol, "p") - n * bs_price(S0, k1, T, r, vol, "p")
        if worst_put is None or credit < worst_put[0]:
            worst_put = (credit, k1, k2, n)
print("smallest call credit: %.4f (sell %d, buy %.2f x %d)" % (worst_call[0], worst_call[1], worst_call[3], worst_call[2]))
print("smallest put credit : %.4f (sell %d, buy %.2f x %d)" % (worst_put[0], worst_put[2], worst_put[3], worst_put[1]))

# cost per delta: the sold option contains more value per delta than the bought ones
print("value per delta point, 95 call: %.4f   105 call: %.4f" % (
    bs_price(S0, 95, T, r, vol) / bs_delta(S0, 95, T, r, vol),
    bs_price(S0, 105, T, r, vol) / bs_delta(S0, 105, T, r, vol)))

## 5. The Figure 11-31 call ratio spread, both sides, with its value before expiration (pages 179, 200)
April options, stock 100, 2 months, 20%, 6%: buy 20 April 110 calls (0.65, delta 16), sell 10 April 105 calls (1.71, delta 33). Delta: $20 \times 16 - 10 \times 33 = -10$. Credit: $10 \times 1.71 - 20 \times 0.65 = 4.10$.

In [ ]:
q110, q105 = 20, -10
print("cash flow:", round(-q110 * 0.65 - q105 * 1.71, 2), "(book +4.10)")
print("delta    :", q110 * 16 + q105 * 33, "(book -10)")
print("gamma    :", round(q110 * 3.0 + q105 * 4.4, 1), "(book +16.0)")
print("theta    :", round(q110 * -0.019 + q105 * -0.030, 3), "(book -0.080)")
print("vega     :", round(q110 * 0.10 + q105 * 0.15, 2), "(book +0.50)")

# value of 2 x 110 calls less 1 x 105 call, today and as time passes (stock volatility 20%, rate 6%)
Sx = np.linspace(85, 130, 226)
fig, ax = plt.subplots(figsize=(7.5, 4))
for T_, lab, st in ((2 / 12, "2 months left", "-"), (1 / 12, "1 month left", "--"), (1e-6, "at expiration", ":")):
    v = curve(lambda s: 2 * bs_price(s, 110, T_, 0.06, 0.20) - bs_price(s, 105, T_, 0.06, 0.20), Sx)
    ax.plot(Sx, v, st, lw=2.5 if T_ > 1 else 1.8, label=lab)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("underlying price"); ax.set_ylabel("value of the position (per 2 x 1)")
ax.set_title("Backspread +2 April 110 calls / -1 April 105 call"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("../images/ch11-ratio-spread-over-time.png", dpi=130); plt.show()

## 6. Christmas tree (pages 182-184)
Long call Christmas tree, April, from the book's table: buy 1 April 100 call (3.76), sell 1 April 105 call (1.71), sell 1 April 110 call (0.65). Debit $= 3.76 - 1.71 - 0.65 = 1.40$. Delta $= 56 - 33 - 16 = +7$.

In [ ]:
debit = 3.76 - 1.71 - 0.65
print("debit:", round(debit, 2), "(book 10 trees = -14.00)  delta:", 56 - 33 - 16, "(book 10 trees = +70)")

Sx = np.linspace(90, 125, 351)
val = call_val(Sx, 100) - call_val(Sx, 105) - call_val(Sx, 110)
pnl = val - debit
print("max value:", val.max(), "  breakevens near", Sx[np.where(np.diff(np.sign(pnl)))[0]].round(2))

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(Sx, call_val(Sx, 100), "--", lw=1, label="+1 x 100 call")
ax.plot(Sx, -call_val(Sx, 105), "--", lw=1, label="-1 x 105 call")
ax.plot(Sx, -call_val(Sx, 110), "--", lw=1, label="-1 x 110 call")
ax.plot(Sx, pnl, "k", lw=3, label="long call Christmas tree, P&L after paying 1.40")
ax.axhline(0, color="grey", lw=0.8); ax.set_ylim(-12, 8)
ax.set_xlabel("underlying price at expiration"); ax.set_ylabel("value of each leg / P&L")
ax.set_title("Long call Christmas tree +1 100 / -1 105 / -1 110"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig("../images/ch11-christmas-tree.png", dpi=130); plt.show()

## 7. Calendar spreads (pages 185-189)
The book's calendar tables (Figures 11-19, 11-20, 11-21) are matched with Black-Scholes using **no interest** (r = 0), 20% volatility and underlying 100 *(our inference: with r = 6% the June 100 call would be 5.62, not 4.60)*. The spread is long 1 June 100 call, short 1 April 100 call.

In [ ]:
def june(S=100, T=4 / 12, vol=0.20): return bs_price(S, 100, T, 0.0, vol)
def april(S=100, T=2 / 12, vol=0.20): return bs_price(S, 100, T, 0.0, vol)

print("Figure 11-19 (time passes, underlying stays 100)")
for months in (0, 1, 2):
    j, a = june(T=(4 - months) / 12), april(T=max(2 - months, 0) / 12)
    print(f"  after {months} month(s): June {j:.2f}  April {a:.2f}  spread {j - a:.2f}")
print("  book: June 4.60 3.99 3.26, April 3.26 2.30 0, spread 1.34 1.69 3.26")

print("Figure 11-20 (underlying changes)")
for s in (80, 90, 100, 110, 120):
    print(f"  S={s}: June {june(S=s):.2f}  April {april(S=s):.2f}  spread {june(S=s) - april(S=s):.2f}")
print("  book spread: 0.09 0.71 1.34 0.88 0.27")

print("Figure 11-21 (volatility changes)")
for v in (0.10, 0.15, 0.20, 0.25, 0.30):
    print(f"  vol {v:.0%}: June {june(vol=v):.2f}  April {april(vol=v):.2f}  spread {june(vol=v) - april(vol=v):.2f}")
print("  book spread: 0.67 1.01 1.34 1.68 2.02")

In [ ]:
# Figure: April and June at-the-money option values and the spread as time passes
ts = np.linspace(0, 2, 201)
jv = np.array([june(T=(4 - t) / 12) for t in ts])
av = np.array([april(T=max(2 - t, 1e-9) / 12) for t in ts])
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(ts, jv, label="June 100 call (long)")
ax.plot(ts, av, label="April 100 call (short)")
ax.plot(ts, jv - av, "k", lw=3, label="long calendar spread = June - April")
ax.scatter([0, 1, 2], [june(T=4/12) - april(T=2/12), june(T=3/12) - april(T=1/12), june(T=2/12)], color="k", zorder=5)
ax.set_xlabel("months passed (underlying stays at 100, volatility 20%)"); ax.set_ylabel("option value")
ax.set_title("At-the-money options: the nearer one decays faster"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("../images/ch11-calendar-time-decay.png", dpi=130); plt.show()

# Figure: spread against the underlying price, and against volatility
fig, axs = plt.subplots(1, 2, figsize=(10.5, 4))
Sx = np.linspace(70, 130, 301)
for months, st in ((0, "-"), (1, "--")):
    sp = curve(lambda s: june(S=s, T=(4 - months) / 12) - april(S=s, T=(2 - months) / 12), Sx)
    axs[0].plot(Sx, sp, st, lw=2.5 if months == 0 else 1.8, label=f"after {months} month(s)" if months else "today")
axs[0].set_title("Calendar spread vs underlying price"); axs[0].set_xlabel("underlying price"); axs[0].set_ylabel("spread value"); axs[0].legend(fontsize=8)
vs = np.linspace(0.05, 0.40, 71)
axs[1].plot(vs * 100, [june(vol=v) for v in vs], label="June 100 call")
axs[1].plot(vs * 100, [april(vol=v) for v in vs], label="April 100 call")
axs[1].plot(vs * 100, [june(vol=v) - april(vol=v) for v in vs], "k", lw=3, label="spread")
axs[1].set_title("Calendar spread vs implied volatility"); axs[1].set_xlabel("volatility (%)"); axs[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("../images/ch11-calendar-underlying-and-vol.png", dpi=130); plt.show()

### Why the calendar spread behaves this way: at-the-money scaling (our short derivation)
For an at-the-money option (Chapter 9): gamma and theta are proportional to $1/\sqrt{T}$, vega is proportional to $\sqrt{T}$. Long the far month and short the near month therefore gives negative gamma ($\tfrac{1}{\sqrt{T_f}} - \tfrac{1}{\sqrt{T_n}} < 0$), positive theta and positive vega.

In [ ]:
S0, K, r, vol = 100, 100, 0.0, 0.20
for label, Tn, Tf in (("April/June", 2 / 12, 4 / 12),):
    gn, vn = bs_gamma_vega(S0, K, Tn, r, vol)
    gf, vf = bs_gamma_vega(S0, K, Tf, r, vol)
    print("gamma near %.4f far %.4f  -> long calendar gamma %+.4f per option" % (gn, gf, gf - gn))
    print("vega  near %.3f far %.3f  -> long calendar vega  %+.3f per vol point" % (vn / 100, vf / 100, (vf - vn) / 100))
    print("ratio far/near: gamma %.3f (1/sqrt ratio %.3f), vega %.3f (sqrt ratio %.3f)" % (
        gf / gn, math.sqrt(Tn / Tf), vf / vn, math.sqrt(Tf / Tn)))

## 8. Futures-market calendar: offsetting with a futures spread (page 191)
Buy 10 March/June call calendar spreads (long 10 June calls, short 10 March calls), calls at the money with delta about 50. Deltas: June $+10 \times 50 = +500$, March $-500$. In a futures market the March and June options have different underlyings, so hedge each month with its own futures.

In [ ]:
june_delta = +10 * 50
march_delta = -10 * 50
print("June options", june_delta, "-> sell", june_delta // 100, "June futures (-500)")
print("March options", march_delta, "-> buy", -march_delta // 100, "March futures (+500)")

## 9. Time butterfly (pages 191-193)
Long time butterfly: buy 2 of the middle expiry, sell 1 each of the nearer and farther expiry, all at the same strike. We use 2, 3 and 4 months, at the money, 20% volatility, r = 0 *(our example)*. It costs a debit, and the ATM scaling says it has negative gamma, positive theta and positive vega.

In [ ]:
def tfly_value(S, vol=0.20):
    return 2 * bs_price(S, 100, 3 / 12, 0, vol) - bs_price(S, 100, 2 / 12, 0, vol) - bs_price(S, 100, 4 / 12, 0, vol)

print("debit at S = 100: %.3f" % tfly_value(100))
g2, v2 = bs_gamma_vega(100, 100, 2 / 12, 0, 0.2)
g3, v3 = bs_gamma_vega(100, 100, 3 / 12, 0, 0.2)
g4, v4 = bs_gamma_vega(100, 100, 4 / 12, 0, 0.2)
print("gamma of the position: %+.4f   vega per vol point: %+.4f" % (2 * g3 - g2 - g4, (2 * v3 - v2 - v4) / 100))
print("value if vol falls to 15%%: %.3f" % tfly_value(100, 0.15))

Sx = np.linspace(75, 125, 251)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(Sx, [tfly_value(s) for s in Sx], "k", lw=3, label="volatility 20%")
ax.plot(Sx, [tfly_value(s, 0.15) for s in Sx], "--", lw=1.8, label="volatility 15%")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("underlying price"); ax.set_ylabel("value of the time butterfly")
ax.set_title("Long time butterfly: +2 x 3-month, -1 x 2-month, -1 x 4-month"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("../images/ch11-time-butterfly.png", dpi=130); plt.show()

## 10. Interest rates and dividends on a stock calendar spread (pages 193-195)
Forward prices with simple interest, stock 100, rate 8%, no dividend: March (3 months) $100 \times (1 + 0.08 \times 0.25) = 102$, June (6 months) $104$. At 10%: $102.5$ and $105$. The June forward moves twice as much, so the June call gains more than the March call and the call calendar spread widens.

Figure 11-28 (stock 100, volatility 20%, March = 3 months, June = 6 months). The interest table uses a 0 dividend. For the dividend table (interest 6%) the book does not say when the dividends are paid; *our reconstruction* is two dividends before June (at 1.5 and 4.5 months) and one before March (at 1.5 months), each discounted at 6% and taken off the stock price. That reproduces every number in the book.

In [ ]:
for rate in (0.08, 0.10):
    print("rate %.0f%%: March forward %.2f  June forward %.2f" % (rate * 100, 100 * (1 + rate * 0.25), 100 * (1 + rate * 0.5)))

print("\ninterest rate     June call  March call  call spread | June put  March put  put spread")
rates = (0, 0.03, 0.06, 0.09, 0.12)
call_sp, put_sp = [], []
for rate in rates:
    jc, mc = bs_price(100, 100, 0.5, rate, 0.2, "c"), bs_price(100, 100, 0.25, rate, 0.2, "c")
    jp, mp = bs_price(100, 100, 0.5, rate, 0.2, "p"), bs_price(100, 100, 0.25, rate, 0.2, "p")
    call_sp.append(jc - mc); put_sp.append(jp - mp)
    print("%5.0f%%            %6.2f     %6.2f      %6.2f   | %6.2f    %6.2f     %6.2f" % (rate * 100, jc, mc, jc - mc, jp, mp, jp - mp))
print("book call spread: 1.65 2.01 2.41 2.83 3.29    put spread: 1.65 1.27 0.94 0.66 0.42")

print("\nquarterly dividend  call spread  put spread")
divs = (0, 0.5, 1.0, 1.5, 2.0)
dcall, dput = [], []
for d in divs:
    S_june = 100 - d * math.exp(-0.06 * 1.5 / 12) - d * math.exp(-0.06 * 4.5 / 12)
    S_march = 100 - d * math.exp(-0.06 * 1.5 / 12)
    jc, jp = bs_price(S_june, 100, 0.5, 0.06, 0.2, "c"), bs_price(S_june, 100, 0.5, 0.06, 0.2, "p")
    mc, mp = bs_price(S_march, 100, 0.25, 0.06, 0.2, "c"), bs_price(S_march, 100, 0.25, 0.06, 0.2, "p")
    dcall.append(jc - mc); dput.append(jp - mp)
    print("%8.2f            %6.2f       %6.2f" % (d, jc - mc, jp - mp))
print("book call spread: 2.41 2.11 1.81 1.54 1.28    put spread: 0.94 1.13 1.32 1.54 1.78")

fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
axs[0].plot([x * 100 for x in rates], call_sp, "o-", label="call calendar spread")
axs[0].plot([x * 100 for x in rates], put_sp, "s-", label="put calendar spread")
axs[0].set_xlabel("interest rate (%)"); axs[0].set_ylabel("spread value"); axs[0].set_title("Rates: calls widen, puts narrow"); axs[0].legend(fontsize=8)
axs[1].plot(divs, dcall, "o-", label="call calendar spread")
axs[1].plot(divs, dput, "s-", label="put calendar spread")
axs[1].set_xlabel("quarterly dividend"); axs[1].set_title("Dividends: the opposite"); axs[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("../images/ch11-calendar-rates-dividends.png", dpi=130); plt.show()

## 11. Greek totals of every example spread (Figure 11-31 / 11-33, pages 198-209)
Option table from Figure 11-31 (stock 100, volatility 20%, rate 6%, no dividend). Each entry is (theoretical value, delta, gamma, theta, vega, rho). A purchase is a cash **debit** (negative), a sale a **credit** (positive). We rebuild every spread total and compare it with the number printed in the book.

In [ ]:
# (value, delta, gamma, theta, vega, rho)
CALL = {
 ("A", 90): (11.17, 93, 1.7, -0.023, 0.06, 0.136), ("A", 95): (6.98, 79, 3.6, -0.031, 0.12, 0.119),
 ("A", 100): (3.76, 56, 4.8, -0.035, 0.16, 0.088), ("A", 105): (1.71, 33, 4.4, -0.030, 0.15, 0.052),
 ("A", 110): (0.65, 16, 3.0, -0.019, 0.10, 0.025),
 ("J", 90): (12.55, 87, 1.8, -0.022, 0.12, 0.249), ("J", 95): (8.71, 75, 2.8, -0.026, 0.18, 0.221),
 ("J", 100): (5.62, 59, 3.4, -0.027, 0.22, 0.178), ("J", 105): (3.36, 42, 3.4, -0.025, 0.23, 0.130),
 ("J", 110): (1.85, 28, 2.9, -0.020, 0.19, 0.086)}
PUT = {
 ("A", 90): (0.27, -7, 1.7, -0.008, 0.06, -0.013), ("A", 95): (1.04, -21, 3.6, -0.016, 0.12, -0.037),
 ("A", 100): (2.77, -44, 4.8, -0.019, 0.16, -0.077), ("A", 105): (5.67, -67, 4.4, -0.013, 0.15, -0.121),
 ("A", 110): (9.56, -84, 3.0, -0.001, 0.10, -0.156),
 ("J", 90): (0.76, -13, 1.8, -0.008, 0.12, -0.045), ("J", 95): (1.83, -25, 2.8, -0.011, 0.18, -0.089),
 ("J", 100): (3.64, -41, 3.4, -0.011, 0.22, -0.148), ("J", 105): (6.28, -58, 3.4, -0.008, 0.23, -0.213),
 ("J", 110): (9.68, -72, 2.9, -0.002, 0.19, -0.274)}

def totals(legs):
    # legs: list of (quantity, month, strike, kind). Returns cash flow, delta, gamma, theta, vega, rho
    out = np.zeros(6)
    for qty, month, strike, kind in legs:
        v, d, g, t, ve, rho = (CALL if kind == "c" else PUT)[(month, strike)]
        out += qty * np.array([-v, d, g, t, ve, rho])   # buying costs cash: sign flips for value only
    return out

# (name, legs, book totals [cash, delta, gamma, theta, vega, rho])
SPREADS = [
 ("Long straddle (Apr)", [(10, "A", 100, "c"), (10, "A", 100, "p")], [-65.30, 120, 96.0, -0.540, 3.20, 0.110]),
 ("Long straddle (Jun, 1 call : 3 puts)", [(10, "J", 95, "c"), (30, "J", 95, "p")], [-142.00, 0, 112.0, -0.590, 7.20, -0.460]),
 ("Short straddle (Apr)", [(-20, "A", 105, "c"), (-10, "A", 105, "p")], [90.90, 10, -132.0, 0.730, -4.50, -0.170]),
 ("Short straddle (Jun)", [(-10, "J", 100, "c"), (-10, "J", 100, "p")], [92.60, -180, -68.0, 0.380, -4.40, -0.300]),
 ("Long strangle (Apr)", [(10, "A", 95, "p"), (10, "A", 110, "c")], [-16.90, -50, 66.0, -0.350, 2.20, -0.120]),
 ("Long strangle (Jun)", [(20, "J", 90, "p"), (10, "J", 110, "c")], [-33.70, -20, 65.0, -0.360, 4.30, -0.040]),
 ("Short strangle (Apr)", [(-30, "A", 100, "p"), (-40, "A", 105, "c")], [151.50, 0, -320.0, 1.770, -10.80, 0.230]),
 ("Short strangle (Jun)", [(-10, "J", 100, "c"), (-10, "J", 105, "p")], [119.00, -10, -68.0, 0.350, -4.50, 0.350]),
 ("Long butterfly (Apr calls)", [(10, "A", 95, "c"), (-20, "A", 100, "c"), (10, "A", 105, "c")], [-11.70, 0, -16.0, 0.090, -0.50, -0.050]),
 ("Long butterfly (Jun puts)", [(10, "J", 90, "p"), (-20, "J", 95, "p"), (10, "J", 100, "p")], [-7.40, -40, -4.0, 0.030, -0.20, -0.150]),
 ("Short butterfly (Apr puts)", [(-10, "A", 100, "p"), (20, "A", 105, "p"), (-10, "A", 110, "p")], [9.90, -60, 10.0, -0.060, 0.40, -0.090]),
 ("Short butterfly (Jun calls)", [(-10, "J", 90, "c"), (20, "J", 95, "c"), (-10, "J", 100, "c")], [7.50, 40, 4.0, -0.030, 0.20, 0.150]),
 ("Long condor (Apr calls)", [(10, "A", 90, "c"), (-10, "A", 95, "c"), (-10, "A", 105, "c"), (10, "A", 110, "c")], [-31.30, -30, -33.0, 0.190, -1.10, -0.100]),
 ("Long condor (Jun puts)", [(10, "J", 95, "p"), (-10, "J", 100, "p"), (-10, "J", 105, "p"), (10, "J", 110, "p")], [-15.90, 20, -11.0, 0.060, -0.80, -0.020]),
 ("Short condor (Apr puts)", [(-10, "A", 90, "p"), (10, "A", 95, "p"), (10, "A", 100, "p"), (-10, "A", 105, "p")], [21.30, 90, 23.0, -0.140, 0.70, 0.200]),
 ("Short condor (Jun calls)", [(-10, "J", 90, "c"), (10, "J", 95, "c"), (10, "J", 105, "c"), (-10, "J", 110, "c")], [23.30, 20, 15.0, -0.090, 1.00, 0.160]),
 ("Call ratio, buy more (Apr)", [(20, "A", 110, "c"), (-10, "A", 105, "c")], [4.10, -10, 16.0, -0.080, 0.50, -0.020]),
 ("Call ratio, buy more (Jun)", [(30, "J", 110, "c"), (-10, "J", 90, "c")], [70.00, -30, 69.0, -0.380, 4.50, 0.090]),
 ("Call ratio, sell more (Apr)", [(20, "A", 95, "c"), (-30, "A", 100, "c")], [-26.80, -100, -72.0, 0.430, -2.40, -0.260]),
 ("Call ratio, sell more (Jun)", [(10, "J", 100, "c"), (-20, "J", 110, "c")], [-19.20, 30, -24.0, 0.130, -1.60, 0.060]),
 ("Put ratio, buy more (Apr)", [(40, "A", 95, "p"), (-10, "A", 110, "p")], [54.00, 0, 114.0, -0.630, 3.80, 0.080]),
 ("Put ratio, buy more (Jun)", [(50, "J", 95, "p"), (-20, "J", 105, "p")], [34.10, -90, 72.0, -0.390, 4.40, -0.190]),
 ("Put ratio, sell more (Apr)", [(10, "A", 105, "p"), (-30, "A", 95, "p")], [-25.50, -40, -64.0, 0.350, -2.10, -0.100]),
 ("Put ratio, sell more (Jun)", [(30, "J", 105, "p"), (-40, "J", 100, "p")], [-42.80, -100, -34.0, 0.020, -1.90, -0.470]),
 ("Long call Christmas tree (Apr)", [(10, "A", 100, "c"), (-10, "A", 105, "c"), (-10, "A", 110, "c")], [-14.00, 70, -26.0, 0.140, -0.90, 0.110]),
 ("Long call Christmas tree (Jun)", [(10, "J", 90, "c"), (-10, "J", 100, "c"), (-10, "J", 110, "c")], [-50.80, 0, -45.0, 0.250, -2.90, -0.150]),
 ("Short call Christmas tree (Apr)", [(-10, "A", 90, "c"), (10, "A", 100, "c"), (10, "A", 105, "c")], [57.00, -40, 75.0, -0.420, 2.50, 0.040]),
 ("Short call Christmas tree (Jun)", [(-10, "J", 95, "c"), (10, "J", 105, "c"), (10, "J", 110, "c")], [35.00, -50, 35.0, -0.190, 2.40, -0.050]),
 ("Long put Christmas tree (Apr)", [(-10, "A", 95, "p"), (-10, "A", 100, "p"), (10, "A", 105, "p")], [-18.60, -20, -30.0, 0.220, -1.30, -0.070]),
 ("Long put Christmas tree (Jun)", [(-10, "J", 90, "p"), (-10, "J", 105, "p"), (10, "J", 110, "p")], [-26.40, -10, -23.0, 0.140, -1.60, -0.160]),
 ("Short put Christmas tree (Apr)", [(10, "A", 95, "p"), (10, "A", 105, "p"), (-10, "A", 110, "p")], [28.50, -40, 50.0, -0.280, 1.70, -0.020]),
 ("Short put Christmas tree (Jun)", [(10, "J", 90, "p"), (10, "J", 95, "p"), (-10, "J", 100, "p")], [10.50, 30, 12.0, -0.080, 0.80, 0.140]),
 ("Long calendar (calls, Jun-Apr 100)", [(10, "J", 100, "c"), (-10, "A", 100, "c")], [-18.60, 30, -14.0, 0.080, 0.60, 0.900]),
 ("Long calendar (puts, Jun-Apr 95)", [(10, "J", 95, "p"), (-10, "A", 95, "p")], [-7.90, -40, -8.0, 0.050, 0.60, -0.520]),
 ("Short calendar (calls 105)", [(10, "A", 105, "c"), (-10, "J", 105, "c")], [16.50, -90, 10.0, -0.050, -0.80, -0.780]),
 ("Short calendar (puts 100)", [(10, "A", 100, "p"), (-10, "J", 100, "p")], [8.70, -30, 14.0, -0.080, -0.60, 0.710]),
]
names = ("cash", "delta", "gamma", "theta", "vega", "rho")
bad = 0
print("%-38s %9s %7s %8s %8s %7s %7s" % ("spread", "cash", "delta", "gamma", "theta", "vega", "rho"))
for name, legs, book in SPREADS:
    t = totals(legs)
    print("%-38s %9.2f %7.0f %8.1f %8.3f %7.2f %7.3f" % ((name,) + tuple(t)))
    for i, (mine, theirs) in enumerate(zip(t, book)):
        if abs(mine - theirs) > 0.0051 * max(1, abs(theirs)) + 1e-9:
            bad += 1
            print("   -> %s: we get %.3f, the book prints %.3f" % (names[i], mine, theirs))
print("\ndisagreements with the book:", bad)
# Expected: four small slips in the printed book (see the chapter note): short straddle (Apr) rho sign,
# long strangle (Jun) delta sign, put ratio sell-more (Jun) theta (0.020 should be 0.200), long put tree (Apr) gamma (-30 should be -40).

## 12. Choosing a spread when implied volatility is low or high (pages 199-208)
Figures 11-32 and 11-33 repeat the option table but with prices set by an implied volatility of **17%** (options cheap) or **23%** (options expensive), while the trader's own volatility is **20%**. For each position, theoretical edge $=\sum \text{quantity} \times (\text{theoretical value} - \text{price})$ with theoretical values at 20% and prices at the implied volatility. The edge should be positive for positive-vega spreads at 17% and for negative-vega spreads at 23%.

In [ ]:
def price_at(month, strike, kind, iv):
    T = 2 / 12 if month == "A" else 4 / 12
    return bs_price(100, strike, T, 0.06, iv, kind)

def edge(legs, iv, my_vol=0.20):
    total = 0.0
    for qty, month, strike, kind in legs:
        T = 2 / 12 if month == "A" else 4 / 12
        theo = bs_price(100, strike, T, 0.06, my_vol, kind)
        total += qty * (theo - price_at(month, strike, kind, iv))
    return total

picks = ["Long straddle (Apr)", "Short straddle (Apr)", "Long strangle (Apr)", "Short strangle (Apr)",
         "Long butterfly (Apr calls)", "Short butterfly (Apr puts)",
         "Call ratio, buy more (Apr)", "Call ratio, sell more (Apr)",
         "Long call Christmas tree (Apr)", "Short call Christmas tree (Apr)"]
lookup = {n: (legs, book) for n, legs, book in SPREADS}
print("%-34s %7s %14s %14s" % ("April spread", "vega", "edge, IV 17%", "edge, IV 23%"))
for name in picks:
    legs, book = lookup[name]
    print("%-34s %7.2f %14.2f %14.2f" % (name, book[4], edge(legs, 0.17), edge(legs, 0.23)))
# a calendar needs the same implied volatility in both months in this simple test
cal = lookup["Long calendar (calls, Jun-Apr 100)"][0]
print("%-34s %7.2f %14.2f %14.2f" % ("Long calendar (calls)", 0.60, edge(cal, 0.17), edge(cal, 0.23)))

## 13. Smaller numbers in the text
**Adjusting by feel (page 202):** underlying 50.00, delta neutral, gamma $-200$ (deltas per point). If the market falls two points, the delta changes by $-200 \times (-2) = +400$.

**A spread order (page 203):** a market maker quotes a straddle 6.25 / 6.75. You sell at the bid 6.25 and buy at the ask 6.75, whatever the split between call and put. The bid-ask width is 0.50.

In [ ]:
gamma, move = -200, -2.00
print("delta after a 2 point fall: %+d" % (gamma * move))
bid, ask = 6.25, 6.75
print("spread width: %.2f; any split such as 3.75+3.00 or 2.00+4.75 gives %.2f" % (ask - bid, 3.75 + 3.00))